# Stage 2 Deep-Dive: Interactive Resume Processing Pipeline

In [11]:
import sys
from pathlib import Path

# Ensure the project's 'src' folder is visible
project_root = Path.cwd()
sys.path.insert(0, str(project_root / "src"))

# Define your CV path (located in the project root folder)
cv_path = project_root / "Нурлан_Нурлыбай_ML.pdf"

print(f"[+] Workspace initialized.")
print(f"[+] Target CV: {cv_path.resolve()}")
print(f"[+] File exists: {cv_path.exists()}")

[+] Workspace initialized.
[+] Target CV: /home/nurlan/projects/ats/Нурлан_Нурлыбай_ML.pdf
[+] File exists: True


## Step 1: Raw Text Extraction

In [12]:
from ats.ingestion.parser.extract import extract_lines

# Extract lines from PDF
lines = extract_lines(cv_path)

print(f"[+] Successfully extracted {len(lines)} lines from the document!\n")
print("--- First 15 lines of your CV: ---")
for idx, line in enumerate(lines):
    print(f"Line {idx:02d}: {repr(line.text)}")

[+] Successfully extracted 46 lines from the document!

--- First 15 lines of your CV: ---
Line 00: 'Нурлан Нурлыбай'
Line 01: 'Астана & Алматы, Казахстан ❙ +7 (708) 669 3696 ❙ n.nurlibay32@gmail.com ❙ https://t.me/NurlanNurlybay'
Line 02: 'ОПЫТ'
Line 03: 'VBox'
Line 04: 'ML Engineer & Data Scientist Июнь 2025 - Настоящее Время'
Line 05: '• Multimodal Product Search (RAG): Разработал систему семантического поиска товаров на'
Line 06: 'базе SigLIP2 и Qwen-VL , достигнув Recall@1 89.16% . Реализовал GPU-планировщик'
Line 07: 'запросов на базе Redis (Round-robin) и оптимизировал потребление VRAM в 2.5 раза за счет'
Line 08: '4-bit квантования.'
Line 09: '• IoT Anti-theft System: Спроектировал гибридную систему детекции краж для Edge-устройств'
Line 10: '(Raspberry Pi). Объединил LSTM для анализа временных последовательностей с XGBoost и'
Line 11: 'Isolation Forest для поиска аномалий в реальном времени, обеспечив баланс между'
Line 12: 'точностью и низкой задержкой (latency).'
Line 13: '•

## Step 2: Resume Segmentation and Section Classification

In [13]:
from ats.ingestion.parser.segment import segment, Section

# Group lines by section boundaries
sections = segment(lines)

print("[+] Segmentation completed! Sections identified:\n")
for section, sec_lines in sections.items():
    print(f"SECTION: {section.value:<12} | Lines found: {len(sec_lines)}\n")
    if sec_lines:
        for line in sec_lines:
            print(line.text)
        print('\n')

[+] Segmentation completed! Sections identified:

SECTION: about        | Lines found: 2

Нурлан Нурлыбай
Астана & Алматы, Казахстан ❙ +7 (708) 669 3696 ❙ n.nurlibay32@gmail.com ❙ https://t.me/NurlanNurlybay


SECTION: experience   | Lines found: 13

VBox
ML Engineer & Data Scientist Июнь 2025 - Настоящее Время
• Multimodal Product Search (RAG): Разработал систему семантического поиска товаров на
базе SigLIP2 и Qwen-VL , достигнув Recall@1 89.16% . Реализовал GPU-планировщик
запросов на базе Redis (Round-robin) и оптимизировал потребление VRAM в 2.5 раза за счет
4-bit квантования.
• IoT Anti-theft System: Спроектировал гибридную систему детекции краж для Edge-устройств
(Raspberry Pi). Объединил LSTM для анализа временных последовательностей с XGBoost и
Isolation Forest для поиска аномалий в реальном времени, обеспечив баланс между
точностью и низкой задержкой (latency).
• Infrastructure & Optimization: Решил проблему утечек памяти (RAM OOM), внедрив
передачу медиаданных через временные

## Step 3: Contact Information and Language Detection

In [14]:
import re
from ats.ingestion.parser.pipeline import _detect_languages, _EMAIL_RE, _PHONE_RE

# Combine all lines to a single string
full_text = "\n".join(line.text for line in lines)

# Extract contact details
email_match = _EMAIL_RE.search(full_text)
email = email_match.group(0) if email_match else "Not Found"

phone_match = _PHONE_RE.search(full_text)
phone = phone_match.group(0) if phone_match else "Not Found"

# Detect languages
languages = _detect_languages(full_text)

print(f"[+] Email:     {email}")
print(f"[+] Phone:     {phone}")
print(f"[+] Languages: {languages}")

[+] Email:     n.nurlibay32@gmail.com
[+] Phone:     +7 (708) 669 3696
[+] Languages: ['ru', 'en']


## Step 4: Named Entity Recognition for Name Extraction

In [15]:
from ats.ingestion.parser.ner import find_name

# Look for name in header lines
name = find_name(lines[:10])

print(f"[+] Name Extracted: {name}")

[+] Name Extracted: Нурлан Нурлыбай


## Step 5: Entity Extraction

In [16]:
from ats.ingestion.parser.ner import extract_entities

# 1. Experience text processing
exp_lines = sections.get(Section.EXPERIENCE, [])
exp_text = "\n".join(line.text for line in exp_lines)
exp_entities = extract_entities(exp_text) if exp_text else []

# 2. Education text processing
edu_lines = sections.get(Section.EDUCATION, [])
edu_text = "\n".join(line.text for line in edu_lines)
edu_entities = extract_entities(edu_text) if edu_text else []

print(f"[+] Experience NLP Entities found: {len(exp_entities)}")
for ent in exp_entities[:10]:
    print(f"  - [{ent.label}] {repr(ent.text)} (chars {ent.start}:{ent.end})")

print(f"\n[+] Education NLP Entities found: {len(edu_entities)}")
for ent in edu_entities[:10]:
    print(f"  - [{ent.label}] {repr(ent.text)}")

[+] Experience NLP Entities found: 39
  - [ORG] 'VBox\nML Engineer & Data Scientist' (chars 0:33)
  - [PERSON] 'Июнь 2025 - Настоящее Время' (chars 34:61)
  - [CARDINAL] '•' (chars 62:63)
  - [ORG] 'Multimodal Product' (chars 64:82)
  - [ORG] 'Product Search' (chars 75:89)
  - [ORG] 'RAG' (chars 91:94)
  - [PERSON] 'Разработал' (chars 97:107)
  - [GPE] 'поиска товаров' (chars 131:145)
  - [PERSON] 'и Qwen-VL' (chars 162:171)
  - [PERCENT] '89.16%' (chars 193:199)

[+] Education NLP Entities found: 2
  - [ORG] 'Назарбаев Университет'
  - [ORG] 'Компьютерные Науки Август 2022 - Июнь 2026'


## Step 6: Structure Reconstruction

In [17]:
from ats.ingestion.parser.pipeline import _build_experience_entries, _build_education_entries

# Reconstruct structured Pydantic lists
experience = _build_experience_entries(exp_lines, exp_entities)
education = _build_education_entries(edu_lines, edu_entities)

print("[+] Reconstructed Work History:")
for i, exp in enumerate(experience, 1):
    print(f"  {i}. {exp.organization or 'Unknown Organization'} | {exp.role or 'Unknown Role'}")
    print(f"     Dates: {exp.start or 'Start'} to {exp.end or 'Present'}")
    if exp.raw:
        # Clean up linebreaks for clean output
        raw_desc = exp.raw.replace('\n', ' ').strip()
        print(f"     Summary: {raw_desc[:90]}...\n")

print("\n[+] Reconstructed Education:")
for i, edu in enumerate(education, 1):
    dates = f"({edu.start} to {edu.end})" if (edu.start or edu.end) else ""
    print(f"  {i}. {edu.institution or 'Unknown Institution'} {dates}")
    if edu.degree:
        print(f"     Degree: {edu.degree}")

[+] Reconstructed Work History:
  1. VBox | ML Engineer & Data Scientist
     Dates: Июнь 2025 to present
     Summary: ML Engineer & Data Scientist Июнь 2025 - Настоящее Время • Multimodal Product Search (RAG)...


[+] Reconstructed Education:
  1. Назарбаев Университет (Август 2022 to Июнь 2026)
     Degree: Компьютерные Науки


## Step 7: Skills Parsing and Fallback Keyword Extraction

In [18]:
from ats.ingestion.parser.skills_section import parse_skills_section
from ats.ingestion.parser.keywords import extract_skills

skill_lines = sections.get(Section.SKILLS, [])
section_skills = parse_skills_section(skill_lines) if skill_lines else []

print(f"[+] Parsed from explicit SKILLS section ({len(section_skills)} skills):")
print(f"  -> {section_skills[:20]}")

# Fallback/complementary KeyBERT keyword extraction
print("\n[*] Running ML KeyBERT keyword extraction on entire CV...")
ml_skills = extract_skills(full_text, exclude_name=name)
print(f"[+] KeyBERT extracted skills ({len(ml_skills)} skills):")
print(f"  -> {ml_skills[:20]}")

[+] Parsed from explicit SKILLS section (27 skills):
  -> ['PyTorch', 'Scikit-learn', 'XGBoost', 'CatBoost', 'Optuna', 'SHAP', 'Evaluation', 'Transformers (HuggingFace)', 'PEFT/LoRA', 'Tokenization', 'Vector Embeddings', 'SigLIP2', 'Qwen-VL', 'RAG', 'Milvus', 'FAISS', 'Python (FastAPI)', 'SQL', 'Bash', 'Docker']

[*] Running ML KeyBERT keyword extraction on entire CV...


/home/nurlan/projects/ats/.venv/lib/python3.14/site-packages/sklearn/feature_extraction/text.py:411: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['ll', 've'] not in stop_words.
  warnings.warn(


[+] KeyBERT extracted skills (15 skills):
  -> ['product search rag', 'vbox ml engineer', 'optimization', 'ai recruiting agent', 'планировщик запросов базе', 'ner ресурсоемких вызовов', 'пайплайн двухстадийного переранжирования', 'инстансов образование назарбаев', 'систему детекции краж', 'vram раза счет', 'разработал систему семантического', 'cloud data qa', 'nlp', 'компьютерные науки август', 'внедрив передачу медиаданных']


## Step 8: BGE-M3 Text Embeddings

In [19]:
from ats.ingestion.parser.embed import embed_text

print("[*] Generating BGE-M3 dense semantic embedding vector (offline CPU)...")
embedding = embed_text(full_text)

print(f"[+] Vector generated successfully!")
print(f"[+] Embedding Dimensions: {len(embedding)}")
print(f"[+] Vector Preview (first 10 components):\n  {embedding[:10]}")

[*] Generating BGE-M3 dense semantic embedding vector (offline CPU)...
[+] Vector generated successfully!
[+] Embedding Dimensions: 1024
[+] Vector Preview (first 10 components):
  [-0.033919695764780045, -0.023846128955483437, -0.01556512713432312, 0.018070921301841736, -0.022825997322797775, 0.051532067358493805, -0.014490935020148754, -0.002577789593487978, -0.024927280843257904, 0.02229754813015461]


## Final Step: Pydantic Assembly

In [20]:
from ats.ingestion.parser.schema import ParsedResume

# Assemble everything together
final_resume = ParsedResume(
    source_file=str(cv_path),
    name=name,
    email=email,
    phone=phone,
    languages_detected=languages,
    skills=section_skills if len(section_skills) >= 3 else ml_skills,
    experience=experience,
    education=education,
    raw_text=full_text,
    embedding=embedding,
)

print("PIPELINE EXECUTION SUCCESSFUL!")
print("==================================================================")
print(f"Candidate Profile: {final_resume.name} ({final_resume.email})")
print(f"Parsed Skills:     {len(final_resume.skills)} tags")
print(f"Parsed Education:  {len(final_resume.education)} entries")
print(f"Parsed Experience: {len(final_resume.experience)} entries")
print("==================================================================")

PIPELINE EXECUTION SUCCESSFUL!
Candidate Profile: Нурлан Нурлыбай (n.nurlibay32@gmail.com)
Parsed Skills:     27 tags
Parsed Education:  1 entries
Parsed Experience: 1 entries
